# Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from os.path import join, exists
import joblib
from tqdm.auto import tqdm
import torch

# Import Functions
sys.path.append("../../")

from src.evaluation.generate_expl import eg_expl_func, ig_expl_func, saliency_expl_func, gc_expl_func, gbp_expl_func
from src.configs.default_configs import device, fn_ue_perf
from src.configs.octmnist_config import data_name, batch_size, data_name_oods
from src.file_manager.filepath import FilePath
from src.data_generator.oct_mnist import load_octmnist_data_dict
from src.data_generator.octdl import load_octdl_data_dict
from src.data_processing.ood_dataset_preprocessing import left_join_datasets
from src.evaluation.generate_expl import get_all_explanations
from torch.nn import Sequential
from src.file_manager.load_save_model import load_model
from src.models.resnet_rue.model import RueResNet18
from src.file_manager.filepath import create_folder

import src.evaluation.road as road
from src.evaluation.road import run_road
road.use_device = device
print("Using device", road.use_device, "for ROAD.")
percentages = [0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9]

import sys
def ignore_child_process_assertion(exctype, value, traceback):
    if exctype is AssertionError and "test a child" in str(value):
        # ignore silently
        return
    # otherwise, use the normal hook
    sys.__excepthook__(exctype, value, traceback)
sys.excepthook = ignore_child_process_assertion

from model_ue_dict import ModelClass_dict, ue_dict
from cur_seed import seed
seed = 2024

fp_notebooks_folder = "./"
fp_project_folder = join(fp_notebooks_folder, "../", "../", "../")
fp_ood2 = FilePath(data_name=data_name_oods[1], seed=seed)

fp = FilePath(data_name=data_name, seed=seed)

fp_road = join(fp.get_parent_folder(fn_ue_perf), "road") 
create_folder(fp_road)

# Load Data

In [ ]:
data_dict = load_octmnist_data_dict(fp_preprocessed=fp.get_preprocessed_folder())
num_ori_test = len(data_dict["test_df"])
octdl_in_data_dict, _ = load_octdl_data_dict(fp_preprocessed=fp_ood2.get_preprocessed_folder())
data_dict = left_join_datasets(data_dict, octdl_in_data_dict)

# Get All Explanations

In [ ]:
expl_func_dict = {"eg": eg_expl_func, "ig": ig_expl_func, "gc": gc_expl_func,"gbp":gbp_expl_func}
expl_dict = {}
for expl_name, expl_func in tqdm(expl_func_dict.items()):
    expl = get_all_explanations(
        expl_name=expl_name, expl_func=expl_func, 
        data_dict=data_dict, split="test", fp=fp, seed=seed, 
        eval_batch_size=16
    )
    expl_dict[expl_name] = expl

In [ ]:
def show_img(ax, dataset, iid, attributions=False):
    import numpy as np
    """ Plot an item of a dataset. Tranfer axis format from channels-first to channels-last."""
    tup = dataset[iid]
    if attributions:
        ax.matshow(np.linalg.norm(tup, axis=2)) # 
    else:
        ax.imshow((tup[0].cpu().transpose(0,1).transpose(1,2)))

def show_image_expl(test_ds, test_expl, img_id=11):
    import matplotlib.pyplot as plt
    f, axes_list = plt.subplots(1, 2)
    show_img(axes_list[0], test_ds, img_id)
    axes_list[0].set_title("input image")

    show_img(axes_list[1], test_expl, img_id, attributions=True)
    axes_list[1].set_title("uncertainty attribution")

    for ax in axes_list:
        ax.set_xticks([])
        ax.set_yticks([])

show_image_expl(test_ds=data_dict["test_df"], test_expl=expl_dict["gbp"], img_id=1000)

# Load Model

In [ ]:
model = load_model(fp=fp, ModelClass=RueResNet18, cur_model_name="tuned")
pred_model = Sequential(model.encoder, model.classifier)

# Run the ROAD Benchmark

In [ ]:
road_dict = {}
for expl_name, expl in tqdm(expl_dict.items()):
    fp_cur = join(fp_road, f"{expl_name}.joblib")
    if not exists(fp_cur):
        road = run_road(
            pred_model, dataset_test=data_dict["test_df"], explanations_test=expl,
            percentages=percentages, morf=True, batch_size=batch_size, transform_test=None)
        joblib.dump(road, fp_cur)
    else:
        road = joblib.load(fp_cur)
    road_dict[expl_name] = road

# Evaluate ROAD

In [ ]:
def plot_road_results(road_dict, percentages):
    import matplotlib.pyplot as plt
    for expl_name, road in road_dict.items():
        plt.plot(percentages, road[0], label=expl_name+"RUE")
    plt.xlabel("% Uncertain Features Removed")
    plt.ylabel("Accuracy")
    plt.legend()

plot_road_results(road_dict, percentages)

In [ ]:
def road_results(road_dict, percentages):
    from sklearn.metrics import auc
    import pandas as pd
    output_df = {}
    for expl_name, road in road_dict.items():
        auroad = auc(percentages, road[0])
        output_df[expl_name+"RUE"] = auroad
    return pd.DataFrame([output_df], index=["Area Under ROAD (↑)"]).round(3)

road_results(road_dict, percentages)